In [ ]:
# GRU-based PFN slice prediction pipeline (Option A - multi-output)
# Notebook-style script. Assumes `df` is already prepared (has PFN, latency_ms, PID, start_ns, etc.)
# Saves models & metadata to ./model/

import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.optimizers import AdamW
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
import math

# -----------------------------
# PARAMETERS (tune these)
# -----------------------------
SEQ_LEN = 30          # number of past steps used to predict next step
BATCH = 64
EPOCHS = 100
MODEL_DIR = 'model_v1'
LR_MAX = 0.001
LR_MIN = 0.0001
WARMUP_EPOCHS = 10
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv("../_filter/filter_swap_log.csv")
# -----------------------------
# 1) Make sure slices exist
# -----------------------------
# Required columns: 'PFN', 'PID', 'start_ns', 'latency_ms'
# Create slice columns if not present
if 'PFN_Top_region' not in df.columns:
    df['PFN_Top_region'] = df['PFN'].apply(lambda x: (int(x) >> 20) & 0xFF)
if 'PFN_slice_4' not in df.columns:
    df['PFN_slice_4'] = df['PFN'].apply(lambda x: (int(x) >> 16) & 0xF)
if 'PFN_slice_3' not in df.columns:
    df['PFN_slice_3'] = df['PFN'].apply(lambda x: (int(x) >> 12) & 0xF)
if 'PFN_slice_2' not in df.columns:
    df['PFN_slice_2'] = df['PFN'].apply(lambda x: (int(x) >> 8) & 0xF)
if 'PFN_slice_1' not in df.columns:
    df['PFN_slice_1'] = df['PFN'].apply(lambda x: (int(x) >> 4) & 0xF)
if 'PFN_slice_0' not in df.columns:
    df['PFN_slice_0'] = df['PFN'].apply(lambda x: int(x) & 0xF)

# normalize latency (if not already)
if 'latency_ms_norm' not in df.columns:
    from sklearn.preprocessing import MinMaxScaler
    lat_s = MinMaxScaler()
    df['latency_ns_norm'] = lat_s.fit_transform(df[['latency_ns']].fillna(0))
    joblib.dump(lat_s, os.path.join(MODEL_DIR, 'scaler_latency.pkl'))

# Ensure proper sorting
group_key = ['PID']
df = df.sort_values(group_key + ['start_ns']).reset_index(drop=True)

# -----------------------------
# 2) Map categorical helpers
# -----------------------------
# PID -> id
df['PID_id'] = df['PID'].astype('category').cat.codes
PID_CNT = int(df['PID_id'].nunique())

# target classes per head
top_classes = sorted(df['PFN_Top_region'].unique())
slice4_classes = sorted(df['PFN_slice_4'].unique())
slice3_classes = sorted(df['PFN_slice_3'].unique())
slice2_classes = sorted(df['PFN_slice_2'].unique())
slice1_classes = sorted(df['PFN_slice_1'].unique())
slice0_classes = sorted(df['PFN_slice_0'].unique())

# mapping from original label -> compact id (0..C-1)
from collections import defaultdict

def make_map(unique_vals):
    m = {v:i for i,v in enumerate(sorted(unique_vals))}
    return m

map_top = make_map(top_classes)
map_4 = make_map(slice4_classes)
map_3 = make_map(slice3_classes)
map_2 = make_map(slice2_classes)
map_1 = make_map(slice1_classes)
map_0 = make_map(slice0_classes)

# convert targets to compact ids
df['y_top'] = df['PFN_Top_region'].map(map_top).astype(int)
df['y_4'] = df['PFN_slice_4'].map(map_4).astype(int)
df['y_3'] = df['PFN_slice_3'].map(map_3).astype(int)
df['y_2'] = df['PFN_slice_2'].map(map_2).astype(int)
df['y_1'] = df['PFN_slice_1'].map(map_1).astype(int)
df['y_0'] = df['PFN_slice_0'].map(map_0).astype(int)

# save metadata
meta = {
    'map_top': map_top, 'map_4': map_4, 'map_3': map_3,
    'map_2': map_2, 'map_1': map_1, 'map_0': map_0,
    'PID_CNT': PID_CNT
}
joblib.dump(meta, os.path.join(MODEL_DIR, 'slice_maps.pkl'))

# -----------------------------
# 3) Build sequence dataset (sliding windows grouped by PID)
# -----------------------------
# features per timestep: slice_4 .. slice_0 (as ints normalized) and latency_ms_norm
# We'll normalize slice integers by dividing by max possible (15) so model sees [0..1]

max_slice_val = 15.0
feature_cols = ['PFN_slice_4','PFN_slice_3','PFN_slice_2','PFN_slice_1','PFN_slice_0','latency_ns_norm']

X_list=[]
pid_list=[]
Y_top=[]
Y_4=[]
Y_3=[]
Y_2=[]
Y_1=[]
Y_0=[]

for pid_id, g in df.groupby('PID_id'):
    arr = g[feature_cols].copy()
    # normalize slices
    for c in ['PFN_slice_4','PFN_slice_3','PFN_slice_2','PFN_slice_1','PFN_slice_0']:
        arr[c] = arr[c].astype(float) / max_slice_val
    arr = arr.values
    # targets
    t_top = g['y_top'].values
    t4 = g['y_4'].values
    t3 = g['y_3'].values
    t2 = g['y_2'].values
    t1 = g['y_1'].values
    t0 = g['y_0'].values

    n = len(g)
    if n <= SEQ_LEN:
        continue
    for i in range(n - SEQ_LEN):
        X_list.append(arr[i:i+SEQ_LEN])
        pid_list.append(pid_id)
        # predict the next step
        j = i + SEQ_LEN
        Y_top.append(t_top[j])
        Y_4.append(t4[j])
        Y_3.append(t3[j])
        Y_2.append(t2[j])
        Y_1.append(t1[j])
        Y_0.append(t0[j])

X = np.array(X_list)
pid_arr = np.array(pid_list).reshape(-1,1)
Y_top = np.array(Y_top)
Y_4 = np.array(Y_4)
Y_3 = np.array(Y_3)
Y_2 = np.array(Y_2)
Y_1 = np.array(Y_1)
Y_0 = np.array(Y_0)

print('Dataset shapes: X', X.shape, 'pid', pid_arr.shape, 'Y_top', Y_top.shape)

# one-hot for classification heads
from tensorflow.keras.utils import to_categorical
Y_top_o = to_categorical(Y_top, num_classes=len(map_top))
Y_4_o = to_categorical(Y_4, num_classes=len(map_4))
Y_3_o = to_categorical(Y_3, num_classes=len(map_3))
Y_2_o = to_categorical(Y_2, num_classes=len(map_2))
Y_1_o = to_categorical(Y_1, num_classes=len(map_1))
Y_0_o = to_categorical(Y_0, num_classes=len(map_0))

# Train/val split
(X_train, X_val,
 pid_train, pid_val,
 ytop_train, ytop_val,
 y4_train, y4_val,
 y3_train, y3_val,
 y2_train, y2_val,
 y1_train, y1_val,
 y0_train, y0_val) = train_test_split(
    X, pid_arr, Y_top_o, Y_4_o, Y_3_o, Y_2_o, Y_1_o, Y_0_o,
    test_size=0.2, random_state=42)

# -----------------------------
# 4) Model building (GRU + PID embedding + multi-head softmax)
# -----------------------------
n_seq_features = X.shape[2]

seq_input = Input(shape=(SEQ_LEN, n_seq_features), name='seq_input')
x = layers.GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3, name='gru_seq')(seq_input)
x = layers.GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, )(x)
x = layers.GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, )(x)
x = layers.GRU(32, return_sequences=False, dropout=0.1, recurrent_dropout=0.1, )(x)

pid_in = Input(shape=(1,), name='pid_in', dtype='int32')
emb_pid = layers.Embedding(input_dim=max(2, PID_CNT), output_dim=16, name='emb_pid')(pid_in)
emb_pid_flat = layers.Flatten()(emb_pid)

combined = layers.Concatenate()([x, emb_pid_flat])
shared = layers.Dense(128, activation='relu')(combined)
shared = layers.Dropout(0.2)(shared)

# heads
head_top = layers.Dense(64, activation='relu')(shared)
out_top = layers.Dense(len(map_top), activation='softmax', name='top_out')(head_top)

head_4 = layers.Dense(64, activation='relu')(shared)
out_4 = layers.Dense(len(map_4), activation='softmax', name='s4_out')(head_4)

head_3 = layers.Dense(64, activation='relu')(shared)
out_3 = layers.Dense(len(map_3), activation='softmax', name='s3_out')(head_3)

head_2 = layers.Dense(64, activation='relu')(shared)
out_2 = layers.Dense(len(map_2), activation='softmax', name='s2_out')(head_2)

head_1 = layers.Dense(64, activation='relu')(shared)
out_1 = layers.Dense(len(map_1), activation='softmax', name='s1_out')(head_1)

head_0 = layers.Dense(64, activation='relu')(shared)
out_0 = layers.Dense(len(map_0), activation='softmax', name='s0_out')(head_0)

model = Model(inputs=[seq_input, pid_in], outputs=[out_top, out_4, out_3, out_2, out_1, out_0])

def lr_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        # linear warmup
        return LR_MIN + (LR_MAX - LR_MIN) * (epoch / WARMUP_EPOCHS)
    # cosine decay after warmup
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    cosine_decay = LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * progress))
    return cosine_decay


lr_callback = LearningRateScheduler(lr_schedule, verbose=1)

optimizer = AdamW(
    learning_rate=LR_MAX, 
    weight_decay=1e-4,
    beta_1=0.9,
    beta_2=0.999
)

model.compile(
    optimizer=optimizer,
    loss={
        'top_out': 'categorical_crossentropy',
        's4_out':  'categorical_crossentropy',
        's3_out':  'categorical_crossentropy',
        's2_out':  'categorical_crossentropy',
        's1_out':  'categorical_crossentropy',
        's0_out':  'categorical_crossentropy',
    },
    metrics={
        'top_out': ['accuracy'],
        's4_out':  ['accuracy'],
        's3_out':  ['accuracy'],
        's2_out':  ['accuracy'],
        's1_out':  ['accuracy'],
        's0_out':  ['accuracy'],
    }
)

model.summary()

# callbacks
callbacks = [
    lr_callback,
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1)
]


Dataset shapes: X (1874, 30, 6) pid (1874, 1) Y_top (1874,)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq_input           │ (None, 30, 6)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_seq (GRU)       │ (None, 30, 128)   │     52,224 │ seq_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_6 (GRU)         │ (None, 30, 64)    │     37,248 │ gru_seq[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pid_in (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_7 (GRU)         │ (None, 30, 64)    │     24,960 │ gru_6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_pid (Embedding) │ (None, 1, 16)     │        240 │ pid_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_8 (GRU)         │ (None, 32)        │      9,408 │ gru_7[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 16)        │          0 │ emb_pid[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 48)        │          0 │ gru_8[0][0],      │
│ (Concatenate)       │                   │            │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 128)       │      6,272 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ top_out (Dense)     │ (None, 13)        │        845 │ dense_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s4_out (Dense)      │ (None, 10)        │        650 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s3_out (Dense)      │ (None, 10)        │        650 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s2_out (Dense)      │ (None, 10)        │        650 │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s1_out (Dense)      │ (None, 10)        │        650 │ dense_19[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s0_out (Dense)      │ (None, 10)        │        650 │ dense_20[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 183,983 (718.68 KB)

 Trainable params: 183,983 (718.68 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    {'seq_input': X_train, 'pid_in': pid_train},
    {'top_out': ytop_train, 's4_out': y4_train, 's3_out': y3_train, 's2_out': y2_train, 's1_out': y1_train, 's0_out': y0_train},
    validation_data=( {'seq_input': X_val, 'pid_in': pid_val}, {'top_out': ytop_val, 's4_out': y4_val, 's3_out': y3_val, 's2_out': y2_val, 's1_out': y1_val, 's0_out': y0_val} ),
    epochs=50,
     batch_size=32,
    callbacks=callbacks,
      verbose=1
)


Epoch 1: LearningRateScheduler setting learning rate to 0.0001.
Epoch 1/50
47/47 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/step - loss: 14.0631 - s0_out_accuracy: 0.0917 - s1_out_accuracy: 0.0972 - s2_out_accuracy: 0.1043 - s3_out_accuracy: 0.1020 - s4_out_accuracy: 0.0914 - top_out_accuracy: 0.2255 - val_loss: 14.0225 - val_s0_out_accuracy: 0.0880 - val_s1_out_accuracy: 0.0987 - val_s2_out_accuracy: 0.0853 - val_s3_out_accuracy: 0.1093 - val_s4_out_accuracy: 0.1200 - val_top_out_accuracy: 0.4053 - learning_rate: 1.0000e-04

Epoch 2: LearningRateScheduler setting learning rate to 0.00019.
Epoch 2/50
47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 13.8846 - s0_out_accuracy: 0.0864 - s1_out_accuracy: 0.0898 - s2_out_accuracy: 0.1009 - s3_out_accuracy: 0.1116 - s4_out_accuracy: 0.1109 - top_out_accuracy: 0.3823 - val_loss: 13.7520 - val_s0_out_accuracy: 0.0933 - val_s1_out_accuracy: 0.1147 - val_s2_out_accuracy: 0.1227 - val_s3_out_accuracy: 0.1120 - val_s4_out_accuracy: 0.1333 - val_top_out_accurac

In [15]:
# save model + metadata
model.save(os.path.join(MODEL_DIR,'gru_slices_full.keras'))
joblib.dump({'SEQ_LEN': SEQ_LEN, 'feature_cols': feature_cols, 'max_slice_val': max_slice_val}, os.path.join(MODEL_DIR,'pipeline_meta.pkl'))

['model_v1\\pipeline_meta.pkl']

In [16]:
# -----------------------------
# 6) Evaluate & reconstruct PFN from predicted slices
# -----------------------------
import numpy as np

preds = model.predict({'seq_input': X_val, 'pid_in': pid_val}, verbose=1)
# preds is list of six arrays
p_top = np.argmax(preds[0], axis=1)
p_4 = np.argmax(preds[1], axis=1)
p_3 = np.argmax(preds[2], axis=1)
p_2 = np.argmax(preds[3], axis=1)
p_1 = np.argmax(preds[4], axis=1)
p_0 = np.argmax(preds[5], axis=1)

# map back to original slice values
inv_map_top = {v:k for k,v in map_top.items()}
inv_map_4 = {v:k for k,v in map_4.items()}
inv_map_3 = {v:k for k,v in map_3.items()}
inv_map_2 = {v:k for k,v in map_2.items()}
inv_map_1 = {v:k for k,v in map_1.items()}
inv_map_0 = {v:k for k,v in map_0.items()}

pred_top_vals = np.array([inv_map_top[int(x)] for x in p_top])
pred_4_vals = np.array([inv_map_4[int(x)] for x in p_4])
pred_3_vals = np.array([inv_map_3[int(x)] for x in p_3])
pred_2_vals = np.array([inv_map_2[int(x)] for x in p_2])
pred_1_vals = np.array([inv_map_1[int(x)] for x in p_1])
pred_0_vals = np.array([inv_map_0[int(x)] for x in p_0])

# true values for comparison (map back)
true_top_vals = np.array([list(map_top.keys())[list(map_top.values()).index(i)] if i in list(map_top.values()) else np.nan for i in np.argmax(ytop_val, axis=1)])
# simpler: derive true directly from df slices (we stored Y_top etc earlier)
true_top_vals = np.array([list(map_top.keys())[list(map_top.values()).index(i)] for i in np.argmax(ytop_val, axis=1)])

# Reconstruct PFN (approx) by combining predicted slices with zero low bits
# pf = (top<<20) | (s4<<16) | (s3<<12) | (s2<<8) | (s1<<4) | s0
pred_recon_pfn = (pred_top_vals.astype(np.int64) << 20) | (pred_4_vals.astype(np.int64) << 16) | (pred_3_vals.astype(np.int64) << 12) | (pred_2_vals.astype(np.int64) << 8) | (pred_1_vals.astype(np.int64) << 4) | pred_0_vals.astype(np.int64)

# To compute true reconstructed PFN for val set, extract from original df rows used for validation
# We built X_val from sliding windows; we have corresponding indices if needed. For now we can rebuild true_recon from y arrays
true_vals_idx = np.argmax(ytop_val, axis=1)
true_4_idx = np.argmax(y4_val, axis=1)
true_3_idx = np.argmax(y3_val, axis=1)
true_2_idx = np.argmax(y2_val, axis=1)
true_1_idx = np.argmax(y1_val, axis=1)
true_0_idx = np.argmax(y0_val, axis=1)

true_top_val = np.array([list(map_top.keys())[list(map_top.values()).index(i)] for i in true_vals_idx])
true_4_val = np.array([list(map_4.keys())[list(map_4.values()).index(i)] for i in true_4_idx])
true_3_val = np.array([list(map_3.keys())[list(map_3.values()).index(i)] for i in true_3_idx])
true_2_val = np.array([list(map_2.keys())[list(map_2.values()).index(i)] for i in true_2_idx])
true_1_val = np.array([list(map_1.keys())[list(map_1.values()).index(i)] for i in true_1_idx])
true_0_val = np.array([list(map_0.keys())[list(map_0.values()).index(i)] for i in true_0_idx])

true_recon_pfn = (true_top_val.astype(np.int64) << 20) | (true_4_val.astype(np.int64) << 16) | (true_3_val.astype(np.int64) << 12) | (true_2_val.astype(np.int64) << 8) | (true_1_val.astype(np.int64) << 4) | true_0_val.astype(np.int64)

# Evaluate region-wise accuracy
acc_top = (pred_top_vals == true_top_val).mean()
acc_4 = (pred_4_vals == true_4_val).mean()
acc_3 = (pred_3_vals == true_3_val).mean()
acc_2 = (pred_2_vals == true_2_val).mean()
acc_1 = (pred_1_vals == true_1_val).mean()
acc_0 = (pred_0_vals == true_0_val).mean()

# Reconstructed PFN MAE
recon_mae = np.mean(np.abs(pred_recon_pfn - true_recon_pfn))

print('Slice accuracies: top,4,3,2,1,0 =', acc_top, acc_4, acc_3, acc_2, acc_1, acc_0)
print('Reconstructed PFN MAE:', recon_mae)

# Save final predictions for inspection
val_out = pd.DataFrame({
    'pred_top': pred_top_vals, 'true_top': true_top_val,
    'pred_s4': pred_4_vals, 'true_s4': true_4_val,
    'pred_s3': pred_3_vals, 'true_s3': true_3_val,
    'pred_s2': pred_2_vals, 'true_s2': true_2_val,
    'pred_s1': pred_1_vals, 'true_s1': true_1_val,
    'pred_s0': pred_0_vals, 'true_s0': true_0_val,
    'pred_recon_pfn': pred_recon_pfn, 'true_recon_pfn': true_recon_pfn
})
val_out.to_csv(os.path.join(MODEL_DIR, 'val_predictions.csv'), index=False)

print('Pipeline finished. Models and metadata saved to', MODEL_DIR)


12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step
Slice accuracies: top,4,3,2,1,0 = 0.02666666666666667 0.06133333333333333 0.07466666666666667 0.12266666666666666 0.10666666666666667 0.104
Reconstructed PFN MAE: 7579634.909333333
Pipeline finished. Models and metadata saved to model_v1
